# Fact transactions
1. read the silver transactions table
2. join with dim_accounts, dim_customer, dim_date
3. select the required columns
- Measures: amount
- Foreign Keys: account_sk, customer_sk, date_key
- Degenerate Dimensions: txn_id, txn_type, channel, status
4. write the transformed data to gold fact_transactions table.

In [0]:
#Imports
from pyspark.sql.functions import col,cast,date_diff

In [0]:
transactions_df = spark.read.table("neo_bank.bronze.transactions")
accounts_df = spark.read.table("neo_bank.gold.dim_accounts")
date_df = spark.read.table("neo_bank.gold.dim_date")

In [0]:

transactions_df = (
    transactions_df.select(
        col("txn_id"),
        col("account_id"),
        col("txn_type"),
        col("amount"),
        col("txn_timestamp").cast("date").alias("txn_date"),
        col("channel"),
        col("status")
    )
)


In [0]:
fact_transactions_df = (
    transactions_df.alias("t")
    .join(
        accounts_df.alias("a"),
        col("t.account_id")==col("a.account_id"),
        "left"
    )
    .join(
        date_df.alias("d"),
        col("t.txn_date")==col("d.full_date"),
        "left"
    )
)

In [0]:
fact_transactions_df = fact_transactions_df.select(
    col("t.txn_id"),
    col("t.txn_type"),
    col("t.channel"),
    col("t.status"),
    col("a.account_sk"),
    col("a.customer_sk"),
    col("d.date_key").alias("txn_date_key"),
    col("t.amount")
)

In [0]:
(
    fact_transactions_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema","True")
        .saveAsTable("neo_bank.gold.fact_transactions")
)

In [0]:
%sql
select * from neo_bank.gold.fact_transactions